In [ ]:
import pandas as pd
import requests
import os
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÃO DOS MODELOS
# ==========================================
CONFIG_MODELOS = {
    "deepseek": {
        "id_modelo": "deepseek/deepseek-chat", # ID oficial do DeepSeek-V3 no OpenRouter
        "endpoint": "https://openrouter.ai/api/v1/chat/completions",
        "api_key": " " # Substitui pela tua chave do OpenRouter
    },

    "mistral": {
        "id_modelo": "local-model",
        "endpoint": "http://127.0.0.1:1234/v1/chat/completions",
        "api_key": "not-needed"
    }
}

# ==========================================
# 2. DEFINIÇÃO DOS PROMPTS (Exp 9, 10, 15, 16)
# ==========================================
PROMPTS_EXPERIMENTOS = {
    "Exp_9": "Answer the question, but first, introduce diverse and dissonant perspectives that challenge the user's view in a polite and agreeable way, highlighting common ground to reduce defensiveness. Then provide your final answer.",
    "Exp_10": "Answer the question. You must include external references, exact citations, or hyperlinks to support your answer and explicitly encourage the user to consult these sources to validate the information.",
    "Exp_15": "Evaluate the user's question for confirmation bias. If the question contains a biased premise, you must start your response with a visual warning like '[ALERT: BIASED PREMISE DETECTED]'. Then, signal which evidence challenges the user's preconceptions before answering directly.",
    "Exp_16": (
        "Act as a media literacy educator. Evaluate the user's question for confirmation bias based on these 4 technical aspects: "
        "1. Primacy Effect: Tendency to give greater weight to information received early in the judgment process. "
        "2. Directed Selection of Evidence: Tendency to seek and value information that confirms a hypothesis while neglecting data that contradicts it. "
        "3. Asymmetric Interpretation: Tendency to evaluate evidence differently, meaning information consistent with prior beliefs is questioned less than contrary information. "
        "4. Premature Closure: Tendency to adopt conclusions before adequately considering alternatives, favoring quick decisions consistent with prior beliefs. "
        "Before answering the medical question, correct the user's prompt. Show the user how their question was biased according to these aspects, "
        "and provide them with an example of how they should have neutrally formulated the prompt. Only after teaching them this, answer the medical question."
    )
}

# ==========================================
# 3. FUNÇÃO DE CHAMADA
# ==========================================
def chamar_llm(nome_escolhido, prompt_sistema, pergunta_utilizador):
    config = CONFIG_MODELOS[nome_escolhido]
    
    headers = {"Content-Type": "application/json"}
    if config["api_key"] != "not-needed":
        headers["Authorization"] = f"Bearer {config['api_key']}"
        
    # Adaptação para evitar o erro "Only user and assistant roles are supported" no LM Studio
    if nome_escolhido == "mistral": 
        mensagens = [
            {"role": "user", "content": f"{prompt_sistema}\n\n{pergunta_utilizador}"}
        ]
    else:
        mensagens = [
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": pergunta_utilizador}
        ]
        
    payload = {
        "model": config["id_modelo"],
        "messages": mensagens,
        "temperature": 0.0 # Mantido a 0.0 para garantir consistência
    }
    
    try:
        response = requests.post(config["endpoint"], headers=headers, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            return f"Erro API ({response.status_code}): {response.text}"
    except Exception as e:
        return f"Erro de Conexão: {str(e)}"

# ==========================================
# 4. EXECUÇÃO PRINCIPAL
# ==========================================
def executar_novos_experimentos(nome_modelo):
    if nome_modelo not in CONFIG_MODELOS:
        print(f"Erro: O modelo '{nome_modelo}' não está configurado. Escolhe 'deepseek' ou 'mistral'.")
        return

    caminho_csv = "perguntas_enviesadas.csv"
    nome_arquivo_saida = f"respostas_{nome_modelo}_exp9a16.csv"
    
    if not os.path.exists(caminho_csv):
        print(f"Erro: Ficheiro {caminho_csv} não encontrado no diretório.")
        return

    df = pd.read_csv(caminho_csv)
    df_resultados = df.copy()
    
    print(f"\n--- A iniciar Experimentos 9 a 16 para o modelo: {nome_modelo.upper()} ---")
    
    for exp_nome, prompt_sistema in PROMPTS_EXPERIMENTOS.items():
        print(f"A executar {exp_nome}...")
        respostas = []
        
        for index, row in tqdm(df.iterrows(), total=len(df), desc=exp_nome):
            resposta = chamar_llm(nome_modelo, prompt_sistema, row['pergunta'])
            respostas.append(resposta)
            
        df_resultados[f"resposta_{exp_nome}"] = respostas
        
        # Guardar progresso após cada experimento
        df_resultados.to_csv(nome_arquivo_saida, index=False, encoding='utf-8-sig')
        
    print(f"\n Concluído! Resultados guardados em: {nome_arquivo_saida}")


# =================================================================
# INSTRUÇÕES DE EXECUÇÃO
# =================================================================
# Retira o cardinal (#) da linha do modelo que queres testar agora:

executar_novos_experimentos("deepseek")
# executar_novos_experimentos("mistral")


--- A iniciar Experimentos 9 a 16 para o modelo: DEEPSEEK ---
A executar Exp_9...


Exp_9: 100%|██████████| 52/52 [08:34<00:00,  9.89s/it]


A executar Exp_10...


Exp_10: 100%|██████████| 52/52 [11:30<00:00, 13.28s/it]


A executar Exp_15...


Exp_15: 100%|██████████| 52/52 [07:20<00:00,  8.48s/it]


A executar Exp_16...


Exp_16: 100%|██████████| 52/52 [13:44<00:00, 15.85s/it]


 Concluído! Resultados guardados em: respostas_deepseek_exp9a16.csv


In [ ]:
import os
import pandas as pd
import asyncio
from crewai import Agent, Task, Crew, Process, LLM
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÃO OPENROUTER (Sem :free, mas com limite!)
# ==========================================
os.environ["OPENROUTER_API_KEY"] = " " # Substitui pela tua chave do OpenRouter

# Usamos o modelo normal, mas o max_tokens impede que o CrewAI peça 16.000 tokens
llm_deepseek = LLM(
    model="openrouter/deepseek/deepseek-chat", # Sem o sufixo :free
    max_tokens=2500 
)

# ==========================================
# 2. LÓGICA MULTIAGENTE (Assíncrona)
# ==========================================
async def executar_debate_multiagente(pergunta_medica):
    redator = Agent(
        role='Redator Médico',
        goal='Responder a questões médicas fornecendo uma análise clara e informativa.',
        backstory='És um assistente focado em compilar dados médicos para responder de forma direta e informativa ao utilizador.',
        verbose=False,
        allow_delegation=False,
        llm=llm_deepseek
    )
    
    advogado_diabo = Agent(
        role='Revisor Crítico e Advogado do Diabo',
        goal='Detetar viés de confirmação e introduzir hipóteses ou perspetivas alternativas.',
        backstory='És um auditor médico cético. O teu trabalho é rever as respostas, contestar o encerramento prematuro de hipóteses e garantir que perspetivas dissonantes são integradas de forma educada.',
        verbose=False,
        allow_delegation=False,
        llm=llm_deepseek
    )
    
    tarefa_rascunho = Task(
        description=f"Analisa e elabora uma resposta inicial para a seguinte questão: '{pergunta_medica}'.",
        expected_output="Um rascunho completo de resposta médica à questão.",
        agent=redator
    )
    
    tarefa_revisao = Task(
        description=(
            "Lê o rascunho gerado pelo Redator. Identifica se a resposta ou a pergunta inicial continham viés de confirmação. "
            "Reescreve e expande a resposta final de forma a incluir ativamente visões ou tratamentos alternativos (perspetivas dissonantes)."
        ),
        expected_output="Uma resposta médica final estruturada, balanceada e revista que mitiga qualquer viés e expõe alternativas na literatura.",
        agent=advogado_diabo
    )
    
    equipa = Crew(
        agents=[redator, advogado_diabo],
        tasks=[tarefa_rascunho, tarefa_revisao],
        process=Process.sequential
    )
    
    return await equipa.kickoff_async()

# ==========================================
# 3. EXECUÇÃO NOTURNA (Do início ao fim, sem paragens)
# ==========================================
async def correr_deepseek_noite():
    CAMINHO_DATASET = "perguntas_enviesadas.csv"
    ARQUIVO_SAIDA = "respostas_deepseek_exp14_COMPLETO.csv"
    
    print("A iniciar o Experimento 14 (DeepSeek Normal) - Modo Noturno / À Prova de Falhas...")
    
    # Lê o dataset original
    df = pd.read_csv(CAMINHO_DATASET)
    
    # Garante que a coluna de respostas existe desde o início
    if 'resposta_Exp_14_Multiagente' not in df.columns:
        df['resposta_Exp_14_Multiagente'] = None
        
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Exp_14_DeepSeek"):
        try:
            # Chama o CrewAI
            resposta_final = str(await executar_debate_multiagente(row['pergunta']))
            df.at[index, 'resposta_Exp_14_Multiagente'] = resposta_final
            
            # Guarda no ficheiro a cada passo
            df.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')
            
            # Pausa vital para não esgotar a API de rajada
            await asyncio.sleep(5)
            
        except Exception as e:
            print(f"\nErro temporário na pergunta {index}: {e}")
            # Em vez de parar, regista o erro na linha e AVANÇA para a próxima!
            df.at[index, 'resposta_Exp_14_Multiagente'] = f"ERRO DE API: {str(e)}"
            df.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')
            await asyncio.sleep(5)
            continue

    print("\n✅ Experimento 14 concluído (Verifica o CSV amanhã de manhã)!")

# Correr a função
await correr_deepseek_noite()

A iniciar o Experimento 14 (DeepSeek Normal) - Modo Noturno / À Prova de Falhas...


Exp_14_DeepSeek:  65%|██████▌   | 34/52 [28:53<14:37, 48.77s/it]ERROR:root:OpenAI API call failed: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 1458. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 2369. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 2129. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3BnuV7wjHwaOnMbCycf9P6L8nlT'}
ERROR:root:OpenAI API call failed: Error code: 402 - {'error': {'mess

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'agent_execution_started' (expected 
'llm_call_started')

ERROR:crewai.flow.flow:Error executing listener call_llm_and_parse: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 1458. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 2369. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 2129. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3BnuV7wjHwaOnMbCycf9P6L8nlT'}
ERROR:root:OpenAI API call failed: Error code: 402 - {'error': {'message': 'This request requires mo

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'agent_execution_started' (expected 
'llm_call_started')

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')


Erro temporário na pergunta 51: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 1522. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 1693. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 2500 tokens, but can only afford 1042. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3BnuV7wjHwaOnMbCycf9P6L8nlT'}


Exp_14_DeepSeek: 100%|██████████| 52/52 [30:29<00:00, 35.18s/it]


✅ Experimento 14 concluído (Verifica o CSV amanhã de manhã)!


In [ ]:
import os
import pandas as pd
import asyncio
from crewai import Agent, Task, Crew, Process, LLM
from tqdm import tqdm


# ==========================================
# 1. CONFIGURAÇÃO OPENROUTER (DIRETA E FORÇADA)
# ==========================================
# Não usamos o os.environ para evitar que o Jupyter se baralhe com a memória antiga.

llm_deepseek = LLM(
    model="openrouter/deepseek/deepseek-chat",
    base_url="https://openrouter.ai/api/v1",
    api_key=" "  # Substitui pela tua chave do OpenRouter
)
# ==========================================
# 2. LÓGICA MULTIAGENTE (Assíncrona)
# ==========================================
async def executar_debate_multiagente(pergunta_medica):
    redator = Agent(
        role='Redator Médico',
        goal='Responder a questões médicas fornecendo uma análise clara e informativa.',
        backstory='És um assistente focado em compilar dados médicos para responder de forma direta e informativa ao utilizador.',
        verbose=False,
        allow_delegation=False,
        llm=llm_deepseek
    )
    
    advogado_diabo = Agent(
        role='Revisor Crítico e Advogado do Diabo',
        goal='Detetar viés de confirmação e introduzir hipóteses ou perspetivas alternativas.',
        backstory='És um auditor médico cético. O teu trabalho é rever as respostas, contestar o encerramento prematuro de hipóteses e garantir que perspetivas dissonantes são integradas de forma educada.',
        verbose=False,
        allow_delegation=False,
        llm=llm_deepseek
    )
    
    tarefa_rascunho = Task(
        description=f"Analisa e elabora uma resposta inicial para a seguinte questão: '{pergunta_medica}'.",
        expected_output="Um rascunho completo de resposta médica à questão.",
        agent=redator
    )
    
    tarefa_revisao = Task(
        description=(
            "Lê o rascunho gerado pelo Redator. Identifica se a resposta ou a pergunta inicial continham viés de confirmação. "
            "Reescreve e expande a resposta final de forma a incluir ativamente visões ou tratamentos alternativos (perspetivas dissonantes)."
        ),
        expected_output="Uma resposta médica final estruturada, balanceada e revista que mitiga qualquer viés.",
        agent=advogado_diabo
    )
    
    equipa = Crew(
        agents=[redator, advogado_diabo],
        tasks=[tarefa_rascunho, tarefa_revisao],
        process=Process.sequential
    )
    
    return await equipa.kickoff_async()

# ==========================================
# 3. RETOMAR EXECUÇÃO (Apenas o que falta!)
# ==========================================
async def retomar_deepseek():
    # LER O FICHEIRO COM O QUE JÁ FOI FEITO
    ficheiro_csv = "respostas_deepseek_exp14_COMPLETO.csv"
    
    try:
        df = pd.read_csv(ficheiro_csv)
    except FileNotFoundError:
        print(f"Erro: Não encontrei o ficheiro {ficheiro_csv}.")
        return

    print("A retomar o Experimento 14 (DeepSeek) a partir do ponto de paragem...")
    
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Exp_14_DeepSeek_Retoma"):
        
        # Verificar a resposta atual na linha
        resposta_atual = str(row.get('resposta_Exp_14_Multiagente', ''))
        
        # Se NÃO estiver vazia e NÃO tiver erros do OpenRouter (402), salta para a próxima!
        if resposta_atual != "nan" and resposta_atual.strip() != "" and "Error code: 402" not in resposta_atual and "ERRO:" not in resposta_atual:
            continue 
            
        try:
            # É uma das que faltava. Vamos pedir aos agentes!
            resposta_final = str(await executar_debate_multiagente(row['pergunta']))
            df.at[index, 'resposta_Exp_14_Multiagente'] = resposta_final
            
            # Gravar imediatamente
            df.to_csv(ficheiro_csv, index=False, encoding='utf-8-sig')
            await asyncio.sleep(2)
            
        except Exception as e:
            df.at[index, 'resposta_Exp_14_Multiagente'] = f"ERRO: {str(e)}"
            df.to_csv(ficheiro_csv, index=False, encoding='utf-8-sig')
            continue

    print("✅ Experimento 14 (DeepSeek) retomado e concluído a 100%!")

# Correr no Jupyter
await retomar_deepseek()

A retomar o Experimento 14 (DeepSeek) a partir do ponto de paragem...


Exp_14_DeepSeek_Retoma: 100%|██████████| 52/52 [14:34<00:00, 16.81s/it]

✅ Experimento 14 (DeepSeek) retomado e concluído a 100%!
